# SignBridge ASL Alphabet Bootstrap

This notebook uses the Kaggle **ASL Alphabet** input (`grassknoted/asl-alphabet`) and the project's `backend/scripts/bootstrap_dataset.py` implementation. It samples up to 400 images from each A-Z folder, ignores `del`, `nothing`, and `space`, and exports only compact 63-feature rows plus metadata and a detection report. It does not export raw images.

**Privacy:** this Kaggle notebook is configured as private in `kernel-metadata.json`; verify its visibility remains **Private** in Kaggle before running or saving versions. No Kaggle API credentials are used. The dataset listing specifies GPL-2, separate from the word dataset's CC BY 4.0 licence.

The project bootstrap script must be available in the private Kaggle notebook environment. The setup cell clones the public project branch without credentials; run this notebook only after that branch contains the reviewed bootstrap implementation. The notebook does not duplicate preprocessing logic.

In [ ]:
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

DATASET_SOURCE = "grassknoted/asl-alphabet"
PROJECT_REPOSITORY = "https://github.com/thenix09/SIGN_BRIDGE.git"
PROJECT_BRANCH = "feature/person-b-ml"
PROJECT_ROOT = Path("/kaggle/working/SIGN_BRIDGE")
BOOTSTRAP_SCRIPT = PROJECT_ROOT / "backend/scripts/bootstrap_dataset.py"

if not BOOTSTRAP_SCRIPT.is_file():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", PROJECT_BRANCH, PROJECT_REPOSITORY, str(PROJECT_ROOT)],
        check=True,
    )
if not BOOTSTRAP_SCRIPT.is_file():
    raise FileNotFoundError(
        "The selected project branch does not contain backend/scripts/bootstrap_dataset.py. "
        "Publish the reviewed script on that branch before running this notebook."
    )
sys.path.insert(0, str(PROJECT_ROOT))
from backend.scripts.bootstrap_dataset import (  # noqa: E402
    MediaPipeHandDetector,
    bootstrap_dataset,
    ensure_task_model,
    find_images_dir,
)

print(f"Project bootstrap: {BOOTSTRAP_SCRIPT}")

## Dataset input

The Kaggle notebook metadata attaches `grassknoted/asl-alphabet` as an input. The project locator checks the standard Kaggle mount and finds the nested A-Z image folders automatically. The raw input remains read-only.

In [ ]:
images_dir = find_images_dir()
print(f"Located alphabet image folders: {images_dir}")
print("Classes used: A-Z; non-letter folders are ignored by the project bootstrap pipeline.")

## Extract and export compact features

The same project bootstrap function used locally runs MediaPipe in static-image mode, canonicalizes handedness using the detected dataset majority, and reuses `HandFrame` for all 63 features. Failed detections and classes with no rows or fewer than 100 rows are reported. Do not interpret the index-based image groups as signer-independent groups.

In [ ]:
export_dir = Path("/kaggle/working/signbridge_alphabet_export")
export_dir.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(prefix="signbridge-alphabet-") as temporary:
    temporary_dir = Path(temporary)
    task_model = ensure_task_model(temporary_dir / "hand_landmarker.task")
    detector = MediaPipeHandDetector(task_model, min_detection_confidence=0.5)
    try:
        report = bootstrap_dataset(
            images_dir=images_dir,
            output_path=temporary_dir / "samples.csv",
            report_path=temporary_dir / "bootstrap_report.json",
            metadata_path=temporary_dir / "letters_meta.json",
            detector=detector,
            per_class=400,
            seed=42,
        )
    finally:
        detector.close()

    for filename in ("samples.csv", "bootstrap_report.json", "letters_meta.json"):
        shutil.copy2(temporary_dir / filename, export_dir / filename)

print(json.dumps({
    "total_tried": report["total_tried"],
    "total_detected": report["total_detected"],
    "total_failed_detections": report["total_failed_detections"],
    "total_written": report["total_written"],
    "missing_classes": report["missing_classes"],
    "classes_below_100_rows": report["classes_below_100_rows"],
    "per_class": report["per_class"],
}, indent=2))
print(f"Compact feature CSV and metadata are in: {export_dir}")

In [ ]:
archive_path = shutil.make_archive(str(export_dir), "zip", root_dir=export_dir)
print(f"Download this compact export from Kaggle notebook output: {archive_path}")
print("The archive contains features and metadata only; no raw images or Kaggle credentials.")